# plot_06. Paper figure candidates

Every candidate figure of the planning notes of 9 October 2026: the panels A to F, the verification against ERA5, and the candidate layouts put together from them. Nothing here is the final choice. The point is to see every candidate on the real data and then pick one; notebook plot_05 keeps the figures already chosen. The regional candidates (A5, B3, C1, C2, R1, R2, D2–D4, E3–E5, F1, V1, V4, and the layouts built on them) were dropped with the regional analysis on 16 November 2026.

Every candidate is a *block* drawn by one of this notebook's `draw_...(fig, ..., tag_start)` functions, into a whole figure or into one part of a layout (`fig.subfigures`), so the same block goes into several layouts: the maps layer by layer with `extant/plots/antarctica.py` (`draw_agreement_maps`, `draw_observed_maps`), the rest on axes from `extant/plots/charts.py` with its shared layers (a point per model and forcing). Every array a panel is given is made here, step by step, from what notebooks calc_02a to calc_02c and calc_03a to calc_03c saved (and notebook calc_04's sea ice): nothing is computed inside a figure.

| | Candidate | From |
|---|---|---|
| A1 | Mean change: the multi-model median with the AR6 hatching | calc_02b §1 `mean_bootstrap` |
| A2 | Mean change: the median, and how many models agree on its sign, per forcing | calc_02b §1 |
| A3 | Mean change, every model, per forcing | calc_02b §1 |
| A4 | Observed against historical: ERA5's trend, historical's, and where ERA5 falls among the members | calc_03a `statistics` (the trend), calc_03c `detections` |
| B1 | Width change: the median with the AR6 hatching, and the change in sea ice (D1) | calc_02a §2 `qrange_change`, calc_04 `climatology` |
| B2 | Width change, every model, per forcing | calc_02a §2 |
| C3 | Probability-ratio maps | calc_02c §1 `probability_ratio` |
| E1, E2 | The SAM: its forced change, and the mean in each SAM phase | calc_02a *SAM Phases* `sam_change`, calc_02b §1 *By SAM phase* |
| F2 | Additivity: historical minus the sum of the single forcings | calc_02b §3 `own_changes`, calc_02a §6 `own_variance_changes` |
| V2, V3, V5 | Verification against ERA5 (Suarez-Gutierrez et al., 2021) | calc_03b §4 `spatial_table`, calc_03a §4–5 `statistics` |
| S | Supplementary: how often each forcing is in each SAM phase, and every model's probability ratios | calc_02a *SAM Phases*, calc_02c §1 |
| L3 | The candidate layout left | all of the above |

**Conventions.** Rows are forcings and columns seasons, as in notebooks plot_03 to plot_05. A multi-model map is the median across the models, with the IPCC AR6 advanced approach to agreement (`config.SIGNIFICANT_THRESHOLD`, `config.SIGN_THRESHOLD`): no hatching where at least two-thirds of the models have a significant change and at least 80% of them agree on its sign (a robust change); diagonal lines where fewer than two-thirds are significant; crossed lines where they are but do not agree on the sign. Each model's significance is its own test against its own internal variability (notebooks calc_02a to calc_02c). The number of models is in each panel's corner; not every model ran every experiment, so a figure that counts models has a colour bar for each experiment, up to that experiment's number of models. Print style (`fr.PAPER`), a page (183 mm) wide, with the caption as the markdown above each figure; every figure is saved as a PDF and a PNG in `figures/<variable>/candidates/`.

**On the sample** everything runs, but on 3×3 cells and two models, so the maps are dots, and the SAM index is a made-up stand-in (notebook calc_02a, *SAM Phases*).

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.patches import Patch

from extant import config, era5_evaluation as ev, response_change as rc
from extant.utils import datatree
from extant.config import ALPHA, FORCED_EXPERIMENTS, SIGNIFICANT_THRESHOLD, SIGN_THRESHOLD
from extant.statistics import significance as sig
from extant.utils.logger import setup_logging
from extant.plots import antarctica, charts, era5_evaluation as evp, forced_response as fr, paper_panels as pp
from plotting_modules.constants import FORCING_COLORS, SEASONS
from plotting_modules import plot_constants

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook; the candidates are
#(t): drawn in fr.PAPER's (STYLE)
plt.rcParams.update(fr.NOTEBOOK.rc)

In [2]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `SIGNIFICANT_THRESHOLD`, `SIGN_THRESHOLD`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
VARIABLE = config.VARIABLES['tas']

#(t): The probability ratios shown: of seasons beyond hist-nat's 95th (warm) or 5th (cold) percentile, which
#(t): hist-nat has a CHANCE of
CHANCE = 0.05

#(t): Print style, a page wide
STYLE = fr.PAPER
WIDTH = fr.PAGE_WIDTH

#(t): The maps of a figure alone: the room above them (inches; their column labels only, as there is no title),
#(t): and their size, four seasons filling the page's width
PAPER_TITLE_HEIGHT = 0.35
MAP_PANEL_SIZE = (WIDTH - antarctica.ROW_LABEL_WIDTH) / len(SEASONS)

In [4]:
#(t): Where notebooks calc_02a to calc_03c and calc_04 saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt')

In [5]:
#(t): The colour scales: notebook plot_03's, and those of the probability ratio, the trends, ERA5's percentile, the change
#(t): in variance, and the share of seasons in a SAM phase
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)
ratio_scale = pp.ratio_scale(f"Probability ratio, beyond hist-nat's {100 * CHANCE:g}th or "
                             f"{100 * (1 - CHANCE):g}th percentile")
trend_scale = fr.Scale(np.round(np.arange(-1, 1.01, 0.2), 2), 'RdBu_r', f'Trend ({VARIABLE.units} per decade)')
percentile_scale = pp.percentile_scale()
variance_scale = fr.Scale(np.round(np.arange(-2, 2.01, 0.4), 2), 'RdBu_r', f'Change in variance ({VARIABLE.units}²)')
PHASE_LABELS = {'negative': 'SAM negative', 'neutral': 'SAM neutral', 'positive': 'SAM positive'}

## Open the Saved Results

Run **one** of the next two sections. Both leave `RESULTS_DIR` and `ICE_RESULTS_DIR`, the folders notebooks calc_02a to calc_03c's and notebook calc_04's results are opened from, and `CANDIDATE_DIR`, where the figures go (this repository's figures folder).

### Sample data

In [6]:
#(t): The sample's results (notebooks calc_02a to calc_03c's, and calc_04's sea ice), and where its figures go
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
CANDIDATE_DIR = Path('../figures') / VARIABLE.name / 'candidates_sample'
RESULTS_DIR, ICE_RESULTS_DIR, CANDIDATE_DIR

(PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/sample'),
 PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/siconc/sample'),
 PosixPath('../figures/tas/candidates_sample'))

### Full data

In [7]:
#(t): The full data's results, and where the candidates go
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
CANDIDATE_DIR = Path('../figures') / VARIABLE.name / 'candidates'
RESULTS_DIR, ICE_RESULTS_DIR, CANDIDATE_DIR

(PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/full'),
 PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/siconc/full'),
 PosixPath('../figures/tas/candidates'))

### Saved results

In [8]:
#(t): The hist-nat bootstrap of the mean change, with hist-nat's and every experiment's final-years mean
mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'mean_bootstrap.zarr').load()

In [9]:
#(t): The width test: the change in Q95 − Q05, and its p-value
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()

In [10]:
#(t): Each experiment's change from its own 1850-1900 baseline
own_changes_da = xr.open_zarr(RESULTS_DIR / 'own_changes.zarr')['own_change'].load()

In [11]:
#(t): Each experiment's change in variance from its own 1850-1900 baseline
own_variance_changes_da = xr.open_zarr(RESULTS_DIR / 'own_variance_changes.zarr')['own_variance_change'].load()

In [12]:
#(t): The probability ratios
pr_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio.zarr').load()

In [17]:
#(t): The forced change of the SAM index
sam_change_da = xr.open_zarr(RESULTS_DIR / 'sam_change.zarr')['sam_change'].load()

In [18]:
#(t): How often each experiment is in each SAM phase
sam_phase_share_da = xr.open_zarr(RESULTS_DIR / 'sam_phase_share.zarr')['phase_share'].load()

In [19]:
#(t): The mean change in each SAM phase
sam_mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'sam_mean_bootstrap.zarr').load()

In [21]:
#(t): Every statistic notebook calc_03a saved, in one store, each a variable named after it: every member's in
#(t): /<model>/historical (on member, season, lat, lon), ERA5's in /ERA5 (on season, lat, lon)
statistics_path = RESULTS_DIR / 'statistics.zarr'
statistics_path

PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/full/statistics.zarr')

In [22]:
statistics_tree = xr.open_datatree(statistics_path, engine='zarr').load()

In [23]:
#(t): The models' members, /<model>/historical
historical_statistics_tree = statistics_tree.match('*/historical')

In [24]:
#(t): ... and ERA5's
era5_statistics_ds = statistics_tree['ERA5'].to_dataset().load()

In [25]:
#(t): The detection classes (notebook calc_03c)
detections_ds = xr.open_zarr(RESULTS_DIR / 'detections.zarr').load()

In [ ]:
#(t): The spatial evaluation (notebook calc_03b)
spatial_table_ds = xr.open_zarr(RESULTS_DIR / 'spatial_table.zarr').load()

In [ ]:
#(t): The number of models with each experiment (not every model ran every experiment), on (experiment): the top of
#(t): each experiment's count colour bar
n_models_da = mean_bootstrap_ds['mean_change'].notnull().any(['season', 'lat', 'lon']).sum('model')
n_models_da.to_pandas()

### Sea ice

The change in sea-ice concentration (notebook calc_04, section 3), for B1 (D1). Run **one** of the next two sections.

#### Without sea ice

In [ ]:
ice_change_da = None
ice_change_median_da = None
ice_change_median_da

#### With sea ice

In [ ]:
#(t): Each experiment's ensemble-mean concentration over its final years, on (model, experiment, season, lat, lon), %
ice_climatology_da = xr.open_zarr(ICE_RESULTS_DIR / 'climatology.zarr')['siconc'].load()

In [ ]:
#(t): The change: each experiment minus hist-nat
ice_change_da = ice_climatology_da - ice_climatology_da.sel(experiment='hist-nat')

In [ ]:
#(t): Its multi-model median, drawn as lines over the width maps (B1): ±5 and ±15%, dashed where ice is lost
ice_change_median_da = ice_change_da.median('model')

## Models Together

The AR6 advanced approach (`sig.model_agreement`) for each change: the median across models, how many of them have data, the share that agree on the sign of the median, and the three exclusive classes, `robust`, `conflicting` and `no_signal`. A model's change counts as significant by its own test in notebooks calc_02a to calc_02c: the hist-nat bootstrap for the mean and for the width, the bootstrap of members for a probability ratio.

In [ ]:
#(t): The mean
mean_agreement_ds = sig.model_agreement(mean_bootstrap_ds['mean_change'], mean_bootstrap_ds['pvalue'] < ALPHA,
                                        sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The width, Q95 - Q05 of the internal variability
width_agreement_ds = sig.model_agreement(qrange_change_ds['qrange_change'], qrange_change_ds['qrange_pvalue'] < ALPHA,
                                         sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The probability ratios as log2 (+1 doubles the chance; a ratio of 0 kept finite)
log_ratio_da = np.log2(pr_ds['pr'].sel(chance=CHANCE).clip(min=2.0 ** -10))

In [ ]:
#(t): and their agreement
ratio_agreement_ds = sig.model_agreement(log_ratio_da, pr_ds['pr_significant'].sel(chance=CHANCE),
                                         sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The legend entries of the maps: AR6's hatching (diagonal lines where no robust change, crossed lines where the
#(t): models conflict), the dots of a single model's maps, and the sea-ice lines of B1. Labels broken onto lines to fit
no_signal_handle = Patch(facecolor='white', edgecolor='0.35', hatch='////',
                         label='No robust change (fewer than\n2/3 of the models significant)')
conflicting_handle = Patch(facecolor='white', edgecolor='0.35', hatch='xxxx',
                           label='Conflicting (2/3 significant, fewer\nthan 80% agree on the sign)')
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none', label='Not significant')
ice_line_handle = plt.Line2D([], [], color='#333333', linewidth=1,
                             label='Change in sea-ice concentration\n(±5, ±15%; dashed: loss)')

In [ ]:
#(t): The agreement maps of A1, B1, C3 and E2, which also go into the layouts at the end
def draw_agreement_maps(fig, agreement_ds, row_dim, colours, row_labels=None, n_models_da=None, extra_handles=(),
                        tag_start=0):
    """The models' median change on (row_dim, season), hatched where they don't agree, in a figure or a layout's part.

    AR6's hatching: diagonal lines where no change is robust, crossed lines where the models conflict. A legend of it
    goes under the median's colour bar.

    Args:
        fig: A figure made with layout='constrained' (from map_figure_size with n_colorbar_rows=1 and n_legends=1),
            or one part of such a layout (fig.subfigures).
        agreement_ds: sig.model_agreement output on (row_dim, season, lat, lon), seasons in SEASONS' order: its
            median, no_signal and conflicting.
        row_dim: The rows' dim, e.g. 'experiment'.
        colours: The median's colours, e.g. scales.change.colours().
        row_labels: The rows' labels; their values if None.
        n_models_da: The number of models of each map, on (row_dim, season), written in its corner; none if None.
        extra_handles: More legend entries, for what the caller draws over the maps (B1's sea-ice lines).
        tag_start: The first map's letter (0 for a).

    Returns:
        The map axes, and the next map's letter (for a layout's next part).
    """
    rows = agreement_ds[row_dim].values
    axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)
    antarctica.heatmaps_of_antarctica(agreement_ds['median'], row_dim, 'season', axes=axes, cax=cax, **colours)
    antarctica.hatching_grid(agreement_ds['no_signal'], row_dim, 'season', axes=axes, pattern='////')
    antarctica.hatching_grid(agreement_ds['conflicting'], row_dim, 'season', axes=axes, pattern='xxxx')
    #(c): The number of models in each map's bottom-right corner
    if n_models_da is not None:
        for i, row in enumerate(rows):
            for j, season in enumerate(SEASONS):
                n_models = float(n_models_da.sel({row_dim: row, 'season': season}))
                axes[i, j].text(1.0, 0.0, f'n = {n_models:g}', transform=axes[i, j].transAxes, ha='right', va='bottom',
                                fontsize=STYLE.tick_size, color='0.3')
    #(c): The legend under the colour bar
    fig.legend(handles=[no_signal_handle, conflicting_handle, *extra_handles], loc='outside lower center',
               ncols=2 + len(extra_handles), frameon=False, fontsize=STYLE.cbar_label_size)

    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, rows if row_labels is None else row_labels, fontsize=STYLE.label_size)
    next_tag = antarctica.tag_panels(axes, start=tag_start, fontsize=STYLE.tag_size)
    return axes, next_tag

The chart blocks of the candidates, each drawn into a figure a page wide or into one part of a layout (`fig.subfigures`), as the agreement maps above: each makes its axes from a GridSpec (`fig.add_gridspec(...).subplots()`), draws its panels with the shared chart layers (or `pp`'s one-axis panels), labels them, letters them from `tag_start`, and returns its axes and the next letter. Each block gives its GridSpec the room round its panels in inches (`charts.margins`), and its legend goes in the room below (`loc='outside lower center'`). The layouts with maps are made with `layout='constrained'`, so that the maps' colour bars get their room under the maps (`antarctica.colorbar_axis_below`); there matplotlib sets the room round the panels itself, and makes room for the legend below them.

In [ ]:
#(t): E1, E4, F1, S1: a change for each forcing (bars: the multi-model median; markers: each model), one row per
#(t): quantity, seasons across
def draw_forcing_bars(fig, rows, x_dim='experiment', bars=FORCED_EXPERIMENTS, tag_start=0):
    """E1, E4, F1 or S1 in a figure (WIDTH wide) or one part of a layout.

    Args:
        rows: Each row's y label -> its values on (model, x_dim, season), top first.
        x_dim: The dim along x, 'experiment', or 'term' for the additivity (with 'sum of parts').
        bars: Its values, in order.
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The room round the panels in inches (charts.margins): above for the titles, below for the turned x labels and
    #(c): the legend
    grid = fig.add_gridspec(len(rows), len(SEASONS), wspace=0.12, hspace=0.2,
                            **charts.margins(fig, top=PAPER_TITLE_HEIGHT, bottom=1.3))
    axes = grid.subplots(sharex=True, sharey='row', squeeze=False)
    for row_axes, (label, values_da) in zip(axes, rows.items()):
        charts.model_dots_grid(values_da.sel({x_dim: bars, 'season': SEASONS}), x_dim, None, 'season', axes=row_axes,
                               color={**FORCING_COLORS, **pp.OTHER_COLORS})
        row_axes[0].set_ylabel(label)
    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left')
    #(c): The bars' names turned, so they fit under a narrow panel
    for ax in axes[-1]:
        for tick_label in ax.get_xticklabels():
            tick_label.set(rotation=40, ha='right')
    models = next(iter(rows.values()))['model'].values
    fig.legend(handles=charts.model_legend_handles(models)
               + [plot_constants.multi_model_median_patch], loc='outside lower center',
               ncols=len(models) + 1, frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): The number of models in each panel
mean_n_models_da = mean_agreement_ds['n_models'].max(['lat', 'lon'])
width_n_models_da = width_agreement_ds['n_models'].max(['lat', 'lon'])
ratio_n_models_da = ratio_agreement_ds['n_models'].max(['lat', 'lon'])
mean_n_models_da.to_pandas()

In [ ]:
#(t): How much of the area south of 60°S each forcing robustly changes (%)
robust_area_df = pd.DataFrame({
    'mean': (100 * sig.area_mean(mean_agreement_ds['robust'].sel(lat=slice(None, -60)))).to_series(),
    'width': (100 * sig.area_mean(width_agreement_ds['robust'].sel(lat=slice(None, -60)))).to_series(),
}).unstack('season').round(0)
robust_area_df

## A. Mean

**A1. The forced change in the mean.** Multi-model median change in the seasonal mean temperature, 1993–2013, against hist-nat over the same years, for each forcing (rows) and season (columns). Hatching after IPCC AR6: diagonal lines, fewer than two-thirds of the models have a significant change (hist-nat bootstrap, p < 0.05); crossed lines, at least two-thirds do but fewer than 80% agree on its sign; none, a robust change. n: the number of models.

In [ ]:
#(t): A1: the models' median change in the mean, every forcing (rows) and season, hatched where they don't agree
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes, next_tag = draw_agreement_maps(fig, mean_agreement_ds.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                         'experiment',
                                         scales.change.colours(), n_models_da=mean_n_models_da)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'A1_mean_agreement')

**A2. The mean change, and how many models agree on its sign.** For each forcing (one figure each): the multi-model median change (top) and the number of models whose change has the sign of the median (bottom).

In [ ]:
#(t): How many models agree on the sign of the median
mean_sign_count_da = (mean_agreement_ds['sign_agreement'] * mean_agreement_ds['n_models']).round()
mean_sign_count_da

In [ ]:
#(t): A2, one figure per forcing: the median (top) and how many models agree on its sign (bottom)
a2_figures = {}
for experiment in FORCED_EXPERIMENTS:
    #(c): The count's colour bar runs up to this experiment's number of models
    count_scale = fr.majority_scale(int(n_models_da.sel(experiment=experiment)),
                                    'Models agreeing on the sign of the median')
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
        #(c): Each row's colour bar under it
        median_cax = antarctica.colorbar_axis_below(axes[0])
        count_cax = antarctica.colorbar_axis_below(axes[1])
        antarctica.heatmaps_of_antarctica(mean_agreement_ds['median'].sel(experiment=experiment, season=SEASONS), None, 'season',
                                          axes=axes[0], cax=median_cax, **scales.change.colours())
        antarctica.heatmaps_of_antarctica(mean_sign_count_da.sel(experiment=experiment, season=SEASONS), None, 'season',
                                          axes=axes[1], cax=count_cax, **count_scale.colours())
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, [f'{experiment}\nmedian', 'Models\nagreeing'], fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    a2_figures[experiment] = fig

In [ ]:
for experiment, fig in a2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'A2_mean_and_sign_{experiment}')

**A3. The mean change in every model.** For each forcing (one figure each): every model's change (rows) in each season. Dots: not significant in that model (hist-nat bootstrap, p ≥ 0.05).

In [ ]:
#(t): Not significant in a model, where it has data
mean_not_significant_da = ~(mean_bootstrap_ds['pvalue'] < ALPHA) & mean_bootstrap_ds['mean_change'].notnull()
mean_not_significant_da

In [ ]:
#(t): A3, one figure per forcing: every model (rows), dotted where not significant
a3_figures = {}
models = mean_bootstrap_ds['mean_change']['model'].values
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                            n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)
        antarctica.heatmaps_of_antarctica(mean_bootstrap_ds['mean_change'].sel(experiment=experiment, season=SEASONS), 'model', 'season', axes=axes,
                                          cax=cax, **scales.change.colours())
        antarctica.not_significant_dots_grid(mean_not_significant_da.sel(experiment=experiment, season=SEASONS), 'model', 'season',
                                             axes=axes)
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, models, fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
        fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=STYLE.cbar_label_size)
    a3_figures[experiment] = fig

In [ ]:
for experiment, fig in a3_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'A3_mean_every_model_{experiment}')

**A4. The observed change against historical.** (a–d) ERA5's trend over the years ERA5 and the models share (1979–2013) and (e–h) the multi-model median of historical's ensemble-mean trend over the same years. (i–l) ERA5's percentile among historical's members (each model's 1979–2013 trends, the median across models): central colours, ERA5 behaves like a member; dark blue or red, ERA5's trend is below or above 95% of the members. Dots: ERA5's trend is outside hist-nat's 5–95% range in at least two-thirds of the models (detected).

In [ ]:
#(t): Each model's historical ensemble mean of every statistic. The trend's is the mean of its members' trends (the
#(t): same as the trend of the ensemble mean, since a straight line through a mean is the mean of the straight lines)
historical_ensemble_mean_tree = historical_statistics_tree.mean('member')

In [ ]:
#(t): The trend's, in one DataArray, on (model, season, lat, lon)
historical_ensemble_mean_ds = datatree.tree_to_dataset(historical_ensemble_mean_tree, dims=['model', 'experiment'])
historical_mean_trend_da = historical_ensemble_mean_ds['trend'].sel(experiment='historical', drop=True)

In [ ]:
#(t): ERA5's trend and historical's multi-model median trend, along a 'source' dim
observed_trend_da = xr.concat([era5_statistics_ds['trend'], historical_mean_trend_da.median('model')],
                              dim=xr.Variable('source', ['ERA5', 'historical (median)']))
observed_trend_da

In [ ]:
#(t): ERA5's percentile among historical's members, the median across models
era5_percentile_da = detections_ds['historical_percentile'].median('model')
era5_percentile_da

In [ ]:
#(t): Where ERA5's trend is outside hist-nat's range (detected: classes 1 and 2) in at least two-thirds of the models
n_detection_models_da = detections_ds['detection_class'].notnull().sum('model')
n_detected_da = detections_ds['detection_class'].isin([1, 2]).sum('model')
detected_da = (n_detected_da >= SIGNIFICANT_THRESHOLD * n_detection_models_da) & (n_detection_models_da > 0)
detected_da

In [ ]:
#(t): A4's maps, which also go into L4: ERA5's trend and historical's median trend (one colour bar), then ERA5's
#(t): percentile among historical's members, dotted where ERA5's trend is detected
def draw_observed_maps(fig, tag_start=0):
    """A4's maps in a figure made with layout='constrained' (from map_figure_size with 3 rows, n_colorbar_rows=2
    and n_legends=1) or one part of such a layout.

    Returns:
        The map axes, and the next map's letter.
    """
    axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
    #(c): The trends' colour bar under rows 1 and 2, the percentile's under row 3
    trend_cax = antarctica.colorbar_axis_below(axes[:2])
    percentile_cax = antarctica.colorbar_axis_below(axes[2])
    antarctica.heatmaps_of_antarctica(observed_trend_da.sel(season=SEASONS), 'source', 'season', axes=axes[:2],
                                      cax=trend_cax, **trend_scale.colours())
    antarctica.heatmaps_of_antarctica(era5_percentile_da.sel(season=SEASONS), None, 'season', axes=axes[2],
                                      cax=percentile_cax, **percentile_scale.colours())
    antarctica.hatching_grid(detected_da.sel(season=SEASONS), None, 'season', axes=axes[2], pattern='....')
    #(c): The legend under the colour bars
    detected_handle = Patch(facecolor='white', edgecolor='0.35', hatch='....',
                            label="Detected: ERA5 outside hist-nat's 5–95% in at least 2/3 of the models")
    fig.legend(handles=[detected_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)

    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, [*observed_trend_da['source'].values, "ERA5's percentile"], fontsize=STYLE.label_size)
    next_tag = antarctica.tag_panels(axes, start=tag_start, fontsize=STYLE.tag_size)
    return axes, next_tag

In [ ]:
#(t): A4 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes, next_tag = draw_observed_maps(fig)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'A4_observed_vs_historical')

## B. Width

**B1. The forced change in width.** Multi-model median change in the width Q95 − Q05 of the internal variability (each member minus its ensemble's forced response), 1993–2013 against hist-nat over the same years. Hatching as in A1 (hist-nat bootstrap, p < 0.05). Lines (with the sea ice, D1): the multi-model median change in sea-ice concentration, ±5 and ±15%, dashed where ice is lost.

In [ ]:
#(t): B1: the models' median change in the width, every forcing (rows) and season, hatched where they don't agree, with
#(t): the median change in sea ice as lines (if the sea ice was opened)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes, next_tag = draw_agreement_maps(fig, width_agreement_ds.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                         'experiment',
                                         scales.spread_change.colours(), n_models_da=width_n_models_da,
                                         extra_handles=[] if ice_change_median_da is None else [ice_line_handle])
    if ice_change_median_da is not None:
        antarctica.contour_lines_grid(ice_change_median_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                      (-15, -5, 5, 15), 'experiment', 'season', axes=axes)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'B1_width_agreement')

**B2. The change in width in every model.** As A3, for the width. Dots: not significant (hist-nat bootstrap, p ≥ 0.05).

In [ ]:
width_not_significant_da = ~(qrange_change_ds['qrange_pvalue'] < ALPHA) & qrange_change_ds['qrange_change'].notnull()
width_not_significant_da

In [ ]:
#(t): B2, one figure per forcing: every model (rows), dotted where not significant
b2_figures = {}
models = qrange_change_ds['qrange_change']['model'].values
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                            n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)
        antarctica.heatmaps_of_antarctica(qrange_change_ds['qrange_change'].sel(experiment=experiment, season=SEASONS), 'model', 'season', axes=axes,
                                          cax=cax, **scales.spread_change.colours())
        antarctica.not_significant_dots_grid(width_not_significant_da.sel(experiment=experiment, season=SEASONS), 'model', 'season',
                                             axes=axes)
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, models, fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
        fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=STYLE.cbar_label_size)
    b2_figures[experiment] = fig

In [ ]:
for experiment, fig in b2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'B2_width_every_model_{experiment}')

## C. Extremes: Probability Ratios

**C3. Probability-ratio maps.** Multi-model median probability ratio of a season warmer than hist-nat's 95th percentile (one figure) or colder than its 5th (the other), 1993–2013 against hist-nat. Orange: more likely; purple: less likely. Hatching as in A1, with significance from the bootstrap (the 95% range of a model's ratio excludes 1).

In [ ]:
#(t): C3, one figure per tail: the models' median probability ratio, every forcing (rows) and season, hatched where they
#(t): don't agree
c3_figures = {}
for tail in ('warm', 'cold'):
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                            n_colorbar_rows=1, n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes, next_tag = draw_agreement_maps(fig, ratio_agreement_ds.sel(tail=tail, experiment=FORCED_EXPERIMENTS,
                                                                         season=SEASONS),
                                             'experiment', ratio_scale.colours(), n_models_da=ratio_n_models_da.sel(tail=tail))
    c3_figures[tail] = fig

In [ ]:
for tail, fig in c3_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'C3_probability_ratio_maps_{tail}')

## E. SAM

From notebooks calc_02a (*SAM Phases*) and calc_02b (section 1's *By SAM phase*). The phases are cut at hist-nat's terciles of the index, so hist-nat has a third of its seasons in each. On the sample the index is made up, so these only show that the figures work.

**E1. The forced change in the SAM.** Each forcing's change in the SAM index, 1993–2013 against hist-nat: the multi-model median (bars) and every model (markers).

In [ ]:
sam_change_da.to_dataframe().unstack('season').round(2)

In [ ]:
#(t): E1 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9))
    axes, _ = draw_forcing_bars(fig, {'Δ SAM index': sam_change_da})

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'E1_sam_change')

**E2. The change in the mean in each SAM phase.** For each forcing (one figure each), the multi-model median change in the mean in the seasons of each SAM phase (rows), 1993–2013 against hist-nat's seasons in the same phase. Hatching as in A1.

In [ ]:
#(t): The AR6 agreement of the change in the mean in each phase
phase_mean_agreement_ds = sig.model_agreement(sam_mean_bootstrap_ds['mean_change'], sam_mean_bootstrap_ds['pvalue'] < ALPHA,
                                              sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)
phase_mean_agreement_ds

In [ ]:
#(t): E2, one figure per forcing: the models' median change in the mean in each SAM phase (rows), hatched where they
#(t): don't agree
e2_figures = {}
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(PHASE_LABELS), n_cols=len(SEASONS),
                                                            n_colorbar_rows=1, n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes, next_tag = draw_agreement_maps(fig, phase_mean_agreement_ds.sel(experiment=experiment, phase=list(PHASE_LABELS),
                                                                              season=SEASONS),
                                             'phase', scales.change.colours(),
                                             row_labels=[f'{experiment}\n{label}' for label in PHASE_LABELS.values()])
    e2_figures[experiment] = fig

In [ ]:
for experiment, fig in e2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'E2_sam_phase_mean_{experiment}')

## F. Synthesis and Additivity

**F2. Where the forcings do not add up.** Multi-model median of historical's change minus the sum of the single forcings' changes, each from its own 1850–1900 baseline: the mean (top) and the variance of the internal variability (bottom). The residual mixes interaction between forcings, forcings left out (land use) and sampling noise.

In [ ]:
#(t): Additivity of the mean (notebook calc_02b, section 3) and of the variance (notebook calc_02a, section 6)
mean_additivity_ds = rc.additivity(own_changes_da)
variance_additivity_ds = rc.additivity(own_variance_changes_da)

In [ ]:
#(t): F2: the models' median residual of the additivity, of the mean (top) and of the variance (bottom)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
    #(c): Each row's colour bar under it
    mean_cax = antarctica.colorbar_axis_below(axes[0])
    variance_cax = antarctica.colorbar_axis_below(axes[1])
    antarctica.heatmaps_of_antarctica(mean_additivity_ds['response'].sel(term='residual', season=SEASONS).median('model'), None,
                                      'season', axes=axes[0], cax=mean_cax, **scales.change.colours())
    antarctica.heatmaps_of_antarctica(variance_additivity_ds['response'].sel(term='residual', season=SEASONS).median('model'),
                                      None, 'season', axes=axes[1], cax=variance_cax, **variance_scale.colours())
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Mean', 'Variance'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'F2_additivity_residual')

## V. Verification Against ERA5

The second link of the attribution chain: historical has to be consistent with ERA5 for the models' forced changes to be credible for the real world (Suarez-Gutierrez et al., 2021), at every grid point. From notebooks calc_03a (sections 4 and 5), calc_03b (section 4) and calc_03c.

**V2. Where each model is adequate.** (Top) How many models are adequate at each grid point (the paper's Fig. 8): ERA5 below every member in fewer than 10% of years, above every member in fewer than 10%, and inside the members' central 75% in at most 80% of them. (Below) Each model: where ERA5 is beyond every member too often, above (red) or below (blue), and hatched where it is in the central 75% too often (too much variability).

In [ ]:
#(t): How many models are adequate at each grid point
adequate_count_ds = ev.adequate_count(spatial_table_ds)
adequate_count_ds

In [ ]:
#(t): Each model's edge class (evp.EDGE_CLASSES)
edge_class_da = evp.edge_class(spatial_table_ds)
edge_class_da

In [ ]:
#(t): V2: how many models are adequate (top), and each model's edge class (rows), hatched where ERA5 is in its
#(t): members' central 75% in more than 80% of years
#(c): The models evaluated against ERA5 (notebook calc_03b), one row each; the count's colour bar runs up to the
#(c): number of them tested
n_eval_models = spatial_table_ds.sizes['model']
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=1 + n_eval_models, n_cols=len(SEASONS),
                                                        n_colorbar_rows=2, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=1 + n_eval_models, n_cols=len(SEASONS))
    #(c): The count's colour bar under row 1, the classes' under the models' rows
    adequate_cax = antarctica.colorbar_axis_below(axes[0])
    edge_cax = antarctica.colorbar_axis_below(axes[1:])
    antarctica.heatmaps_of_antarctica(adequate_count_ds['n_adequate'].sel(season=SEASONS), None, 'season', axes=axes[0],
                                      cax=adequate_cax,
                                      **fr.majority_scale(int(adequate_count_ds['n_tested'].max()),
                                                          'Adequate models').colours())
    antarctica.heatmaps_of_antarctica(edge_class_da.sel(season=SEASONS), 'model', 'season', axes=axes[1:], cax=edge_cax,
                                      **pp.class_scale(evp.EDGE_CLASSES, evp.EDGE_COLORS,
                                                       '% of years ERA5 is beyond every member', wrap=8).colours())
    antarctica.hatching_grid((spatial_table_ds['central_flag'] == 1).sel(model=edge_class_da['model'], season=SEASONS),
                             'model', 'season', axes=axes[1:], pattern='////')
    #(c): The legend under the colour bars
    central_handle = Patch(facecolor='white', edgecolor='0.35', hatch='////',
                           label="ERA5 in the members' central 75% in more than 80% of years")
    fig.legend(handles=[central_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Adequate\nmodels', *edge_class_da['model'].values], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'V2_spatial_evaluation')

**V3. What is wrong, and is it the forcing or the variability?** Each model's diagnosis over 1979–2013 (left of the colour bar: adequate) and, for each problem, whether it is the same in 1979–1996 and 1997–2013 (the variability) or changes between them (the forced response).

In [ ]:
#(t): V3: each model's diagnosis (top rows) and its early-against-late diagnosis (bottom rows)
#(c): The models evaluated against ERA5 (notebook calc_03b); colour-bar rows of 1 in, for the classes' names on three lines
models = spatial_table_ds['model'].values
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(models), n_cols=len(SEASONS), n_colorbar_rows=2,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT,
                                                        colorbar_height=1.0), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2 * len(models), n_cols=len(SEASONS))
    #(c): The diagnoses' colour bar under their rows, early against late's under theirs
    diagnosis_cax = antarctica.colorbar_axis_below(axes[:len(models)])
    period_cax = antarctica.colorbar_axis_below(axes[len(models):])
    antarctica.heatmaps_of_antarctica(spatial_table_ds['diagnosis'].sel(season=SEASONS), 'model', 'season',
                                      axes=axes[:len(models)], cax=diagnosis_cax,
                                      **pp.class_scale(list(ev.DIAGNOSES.values()), evp.DIAGNOSIS_COLORS,
                                                       'Diagnosis', wrap=10).colours())
    antarctica.heatmaps_of_antarctica(spatial_table_ds['period_diagnosis'].sel(season=SEASONS), 'model', 'season',
                                      axes=axes[len(models):], cax=period_cax,
                                      **pp.class_scale(list(ev.PERIOD_DIAGNOSES.values()), evp.PERIOD_COLORS,
                                                       'Early against late').colours())
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, [*models, *[f'{model}\nearly v late' for model in models]], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'V3_diagnosis')

**V5. The moment test over the whole map** (worked out here from notebook calc_03a's values, as in plot_01c's section 7), for the supplement as a table: ✗ where ERA5 is flagged over more of the map than a perfect model is 90% of the time.

In [ ]:
#(t): The four moments at every grid point: where ERA5's falls among each model's members, point by point
#(t): (ev.locate), along a statistic dim
moment_statistics = ['mean', 'std', 'skewness', 'kurtosis']
moment_map_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node.to_dataset()[moment_statistics].to_dataarray('statistic'),
                         era5_statistics_ds[moment_statistics].to_dataarray('statistic'))
    for node in historical_statistics_tree.leaves
})

In [ ]:
#(t): The field test of every model, moment and season (ev.field_test, step by step in plot_01c), without the members
field_test_tree = xr.DataTree.from_dict({
    node.path: ev.field_test(node.to_dataset()).drop_dims('member') for node in moment_map_test_tree.leaves
})

In [ ]:
#(t): On (model, statistic, season)
eval_map_table_ds = datatree.tree_to_dataset(field_test_tree, dims=['model', 'experiment']).sel(experiment='historical',
                                                                                                 drop=True)

In [ ]:
field_verdict_df = eval_map_table_ds['field_verdict'].sel(season=SEASONS).to_series().unstack(['statistic', 'season'])
field_verdict_df.replace({0.0: '✓', 1.0: '✗'})

## S. Supplementary

**S1. How often each SAM phase.** The share of each forcing's seasons in the negative (top) and the positive SAM phase (bottom), 1993–2013: a third in hist-nat, by construction. The multi-model median (bars) and every model (markers).

In [ ]:
#(t): S1 alone: the share of each forcing's seasons in the negative (top) and positive SAM phase
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 4.5))
    axes, _ = draw_forcing_bars(fig, {f'Share of seasons, {PHASE_LABELS[phase]}': sam_phase_share_da.sel(phase=phase)
                                      for phase in ('negative', 'positive')})

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'S1_sam_phase_share')

**S2. Every model's probability ratios.** As C3 for historical, every model (rows); dots: the 95% bootstrap range includes 1.

In [ ]:
#(t): historical's ratios in every model, and where they are not significant
model_ratio_da = log_ratio_da.sel(experiment='historical')
model_ratio_not_significant_da = ~pr_ds['pr_significant'].sel(experiment='historical', chance=CHANCE) \
    & model_ratio_da.notnull()
model_ratio_not_significant_da

In [ ]:
#(t): S2, one figure per tail: historical's probability ratio in every model (rows), dotted where not significant
s2_figures = {}
models = model_ratio_da['model'].values
for tail in ('warm', 'cold'):
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                            n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)
        antarctica.heatmaps_of_antarctica(model_ratio_da.sel(tail=tail, season=SEASONS), 'model', 'season', axes=axes,
                                          cax=cax, **ratio_scale.colours())
        antarctica.not_significant_dots_grid(model_ratio_not_significant_da.sel(tail=tail, season=SEASONS), 'model', 'season',
                                             axes=axes)
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, models, fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
        fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=STYLE.cbar_label_size)
    s2_figures[tail] = fig

In [ ]:
for tail, fig in s2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'S2_probability_ratios_every_model_{tail}')

## Candidate Layouts

The blocks above, put together (`fig.subfigures`, one part per block), the panel letters running on from block to block. A layout figure that is a single candidate is not drawn again. The plan of 9 October:

| Layout | Fig. 1 | Fig. 2 | Fig. 3 | Fig. 4 | Fig. 5 |
|---|---|---|---|---|---|
| L1 current plan | A1 + C1 | B1 + C2 | D2 | E2 + E3 | |
| L2 merged mechanism | A1 + C1 | B1 + C2 | D2, D3 / E1, E2 | F1 | |
| L3 original outline | A2 (historical) | the width as A2 | A1 | B1 | F2 |
| L4 obs-led | A4 + A5 | A1 | B1 or B3 | F1 | D2 + E2 |
| L5 short NCC | the strongest result (a candidate: R1 + C2) | A1 or F1 | D2 + E3 | C2 | |

Every layout with a regional block (A5, B3, C1, C2, R1, D2, D3, E3) was dropped with the regional analysis; L3's Fig. 2 is the one left to draw.

**L3, Fig. 2: the width, as A2.** For historical: the median change in width, and how many models agree on its sign.

In [ ]:
#(t): How many models agree on the sign of the median change in width
width_sign_count_da = (width_agreement_ds['sign_agreement'] * width_agreement_ds['n_models']).round()
width_sign_count_da

In [ ]:
#(t): L3, figure 2: historical's median change in the width (top) and how many models agree on its sign (bottom)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
    #(c): Each row's colour bar under it
    median_cax = antarctica.colorbar_axis_below(axes[0])
    count_cax = antarctica.colorbar_axis_below(axes[1])
    antarctica.heatmaps_of_antarctica(width_agreement_ds['median'].sel(experiment='historical', season=SEASONS), None, 'season',
                                      axes=axes[0], cax=median_cax, **scales.spread_change.colours())
    antarctica.heatmaps_of_antarctica(width_sign_count_da.sel(experiment='historical', season=SEASONS), None, 'season',
                                      axes=axes[1], cax=count_cax,
                                      **fr.majority_scale(int(width_n_models_da.sel(experiment='historical').max()),
                                                          'Models agreeing on the sign of the median').colours())
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['historical\nmedian', 'Models\nagreeing'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L3_fig2_width_historical')

## What Was Saved

In [ ]:
pd.Series(sorted(path.name for path in CANDIDATE_DIR.glob('*.pdf')), name=str(CANDIDATE_DIR)).to_frame()